# Data Cleaning 09 -- WRDS Macro Monthly

## Input
`Data/Data_Collection/Initial/09_Macro_Daily_Monthly_WRDS/macro_monthly.parquet` (252 rows, 22 factor columns, 2004--2024)

## Purpose
Cleans monthly macro/market-level data from WRDS. Contains Treasury bond returns and index levels across the maturity curve (1Y, 2Y, 5Y, 7Y, 10Y, 20Y, 30Y), T-bill returns (30-day, 90-day), CPI return and index, and Pastor-Stambaugh liquidity factors. Keyed on date only (no PERMNO).

## Stage 0: Load & Inspect
Basic shape, date range, column inventory, and dtype verification. Date frequency check confirms dates are the last trading day of each month (70.2% land on the calendar month-end, the rest on the last Friday). Rows per year verified at exactly 12 for all years.

## Stage 1: Missing Data Audit
- Total NaN count: zero
- Per-column NaN: zero across all 22 factors
- Per-row NaN: zero across all 252 rows

## Stage 2: Placeholder Value Detection
- Common placeholder values (-99, -999, etc.): none found
- Suspiciously repeated exact values: none found
- Excessive zeros: none found
- Extreme outliers (>10 sigma): none found

## Stage 3: Per-Factor Gap Analysis
No gaps to report -- zero NaN across all factors.

## Stage 4: Value Range & Quality Checks
- Full summary statistics for all factor columns
- Scale check (percentage vs decimal)
- Constant or near-constant column check: none found
- Overlap with macro_daily: none. Entirely different variables (Treasury/CPI/liquidity vs VIX/FF/FX/world indices).

## Stage 6: Clean & Save

### No Columns Dropped
All 22 factors retained.

### No Rows Dropped
Perfect monthly coverage: 252 rows (21 years x 12 months).

### Zero NaN
No missing data, no placeholders, no constant columns.

### Dates Left As-Is
Dates are the last trading day of the month. The merge pipeline aligns via `merge_asof(direction='backward')`.

### No Winsorisation
Applied in the merge pipeline.

### No Overlap with Macro Daily
Entirely different variables -- no deduplication needed.

## Output
`Data/Data_Collection/Cleaned/09_Macro_Daily_Monthly_WRDS/macro_monthly_clean.parquet` -- 22 factor columns (all retained), 252 rows

In [1]:
# %% [markdown]
# # Data Cleaning: macro_monthly.parquet
#
# Source: Data/Data_Collection/Initial/09_Macro_Daily_Monthly_WRDS/macro_monthly.parquet
# Output: Data/Data_Collection/Cleaned/09_Macro_Daily_Monthly_WRDS/macro_monthly_clean.parquet
#
# Monthly macro/market-level data from WRDS. Keyed on date only (no PERMNO).
# Likely contains monthly versions of Fama-French factors, market aggregates,
# and other macro variables not available at daily frequency.

# %%
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path('../../../Data/Data_Collection/Initial/09_Macro_Daily_Monthly_WRDS/macro_monthly.parquet')
OUT_DIR  = Path('../../../Data/Data_Collection/Cleaned/09_Macro_Daily_Monthly_WRDS')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 0: LOAD & INSPECT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 0: LOAD & INSPECT — Macro Monthly")
print("=" * 90)

df = pd.read_parquet(RAW_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

factor_cols = [c for c in df.columns if c != 'date']

print(f"\n  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  Unique dates: {df['date'].nunique():,}")
print(f"  Factor columns: {len(factor_cols)}")

print(f"\nColumns and dtypes ({len(factor_cols)} factors):")
for i, c in enumerate(factor_cols, 1):
    print(f"  {i:>3d}. {c:<35s} {str(df[c].dtype):<15s}")

print(f"\n--- Head (10 rows) ---")
print(df.head(10).to_string(index=False))

print(f"\n--- Tail (10 rows) ---")
print(df.tail(10).to_string(index=False))

# ── Date frequency check ────────────────────────────────────────────────────
print(f"\n--- Date frequency ---")
is_month_end = df['date'].dt.is_month_end
print(f"  Dates that are month-end: {is_month_end.sum():,} ({is_month_end.mean()*100:.1f}%)")

date_diffs = df['date'].diff().dt.days.dropna()
print(f"  Gap distribution (days):")
print(f"    Mean: {date_diffs.mean():.1f}, Median: {date_diffs.median():.0f}")
print(f"    Min: {date_diffs.min():.0f}, Max: {date_diffs.max():.0f}")

# ── Rows per year ────────────────────────────────────────────────────────────
print(f"\n--- Rows per year ---")
rows_per_year = df.groupby(df['date'].dt.year).size()
for year, n in rows_per_year.items():
    flag = "  ⚠" if n != 12 else ""
    print(f"  {year}: {n:>3d} months{flag}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1: MISSING DATA AUDIT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 1: MISSING DATA AUDIT")
print("=" * 90)

n_rows = len(df)

# ── Total NaN ────────────────────────────────────────────────────────────────
total_cells = n_rows * len(factor_cols)
total_nan = df[factor_cols].isna().sum().sum()
print(f"\nTotal cells: {total_cells:,}")
print(f"Total NaN:   {total_nan:,} ({total_nan / total_cells * 100:.2f}%)")

# ── Per-column NaN with first/last valid dates ───────────────────────────────
col_nan = df[factor_cols].isna().sum()
col_nan_pct = (col_nan / n_rows * 100).round(2)
col_nan_sorted = col_nan_pct.sort_values(ascending=False)

print(f"\n--- Per-Column NaN ---")
print(f"\n  {'Column':<35s} {'NaN %':>8s}  {'Count':>6s}  {'First Valid':>12s}  {'Last Valid':>12s}")
print("  " + "-" * 80)
for col, pct in col_nan_sorted.items():
    count = int(col_nan[col])
    valid = df[df[col].notna()]['date']
    first = valid.min().date() if len(valid) > 0 else 'N/A'
    last = valid.max().date() if len(valid) > 0 else 'N/A'
    flag = " ← DROP" if pct >= 30 else (" ← INVESTIGATE" if pct >= 10 else "")
    print(f"  {col:<35s} {pct:>7.2f}%  {count:>6d}  {str(first):>12s}  {str(last):>12s}{flag}")

# ── Per-row NaN ──────────────────────────────────────────────────────────────
row_nan = df[factor_cols].isna().sum(axis=1)
print(f"\n--- Per-Row NaN Distribution ---")
print(f"  Rows with 0 NaN: {(row_nan == 0).sum():>6,d} ({(row_nan == 0).mean()*100:.1f}%)")
print(f"  Rows with 1-3 NaN: {((row_nan >= 1) & (row_nan <= 3)).sum():>6,d}")
print(f"  Rows with 4+ NaN: {(row_nan >= 4).sum():>6,d}")
print(f"  Max NaN in any row: {row_nan.max()} out of {len(factor_cols)}")

if (row_nan > 0).any():
    print(f"\n  5 rows with most NaN:")
    worst = df.loc[row_nan.nlargest(5).index, ['date']].copy()
    worst['n_nan'] = row_nan.nlargest(5).values
    print(worst.to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2: PLACEHOLDER VALUE DETECTION
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 2: PLACEHOLDER VALUE DETECTION")
print("=" * 90)

placeholders = [-99, -999, -9999, 99, 999, 9999, -99.99, -999.99, 99.99, 999.99]

print(f"\n--- Checking for common placeholder values ---")
any_found = False
for val in placeholders:
    hits = {}
    for col in factor_cols:
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue
        n = (df[col] == val).sum()
        if n > 0:
            hits[col] = n
    if hits:
        any_found = True
        print(f"\n  Value = {val}:")
        for col, n in sorted(hits.items(), key=lambda x: -x[1]):
            pct = n / len(df[col].dropna()) * 100
            print(f"    {col:<35s} {n:>4d} occurrences ({pct:.2f}%)")

if not any_found:
    print(f"  ✓ No common placeholders found")

# ── Suspiciously repeated values ─────────────────────────────────────────────
print(f"\n--- Columns with suspiciously common exact values ---")
for col in factor_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        continue
    vals = df[col].dropna()
    if len(vals) == 0:
        continue
    top_val = vals.value_counts().iloc[0]
    top_pct = top_val / len(vals) * 100
    if top_pct > 20:
        most_common = vals.value_counts().index[0]
        print(f"  {col:<35s} value {most_common:>12.4f} appears {top_val:,} times ({top_pct:.1f}%)")

# ── Excessive zeros ──────────────────────────────────────────────────────────
print(f"\n--- Columns with excessive zeros ---")
any_zeros = False
for col in factor_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        continue
    vals = df[col].dropna()
    if len(vals) == 0:
        continue
    n_zero = (vals == 0).sum()
    pct_zero = n_zero / len(vals) * 100
    if pct_zero > 30:
        any_zeros = True
        print(f"  {col:<35s} {n_zero:,} zeros ({pct_zero:.1f}%)")

if not any_zeros:
    print(f"  ✓ No columns with excessive zeros")

# ── Extreme outliers ─────────────────────────────────────────────────────────
print(f"\n--- Potential outlier placeholders (values >10 std from mean) ---")
for col in factor_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        continue
    vals = df[col].dropna()
    if len(vals) < 20:
        continue
    mean = vals.mean()
    std = vals.std()
    if std == 0:
        continue
    n_extreme = (((vals - mean).abs() / std) > 10).sum()
    if n_extreme > 0:
        extremes = vals[((vals - mean).abs() / std) > 10]
        print(f"  {col:<35s} {n_extreme:>3d} values >10σ  "
              f"(range: [{extremes.min():.4f}, {extremes.max():.4f}], "
              f"mean: {mean:.4f}, std: {std:.4f})")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3: PER-FACTOR GAP ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 3: PER-FACTOR GAP ANALYSIS")
print("=" * 90)

print(f"\n--- Longest consecutive NaN runs per factor ---")
print(f"\n  {'Factor':<35s} {'Max Run':>8s}  {'Total NaN':>10s}  {'NaN %':>7s}  {'Where'}")
print("  " + "-" * 85)

for col in factor_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        continue
    is_nan = df[col].isna()
    total_nan_col = is_nan.sum()
    if total_nan_col == 0:
        continue

    runs = is_nan.ne(is_nan.shift()).cumsum()
    nan_runs = is_nan.groupby(runs).sum()
    nan_runs = nan_runs[nan_runs > 0]

    if len(nan_runs) == 0:
        continue

    max_run = int(nan_runs.max())
    max_run_idx = nan_runs.idxmax()

    run_rows = df[runs == max_run_idx]
    run_start = run_rows['date'].iloc[0].date()
    run_end = run_rows['date'].iloc[-1].date()

    pct = total_nan_col / n_rows * 100
    print(f"  {col:<35s} {max_run:>8d}  {total_nan_col:>10,d}  {pct:>6.2f}%  {run_start} → {run_end}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4: VALUE RANGE & QUALITY CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 4: VALUE RANGE & QUALITY CHECKS")
print("=" * 90)

print(f"\n--- Summary statistics ---")
print(f"\n  {'Column':<35s} {'min':>12s}  {'median':>12s}  {'max':>12s}  {'mean':>12s}  {'std':>12s}")
print("  " + "-" * 100)
for col in factor_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        print(f"  {col:<35s} (non-numeric)")
        continue
    vals = df[col].dropna()
    if len(vals) == 0:
        print(f"  {col:<35s} (all NaN)")
        continue
    print(f"  {col:<35s} {vals.min():>12.4f}  {vals.median():>12.4f}  "
          f"{vals.max():>12.4f}  {vals.mean():>12.4f}  {vals.std():>12.4f}")

# ── Scale check ──────────────────────────────────────────────────────────────
print(f"\n--- Scale check: percentage vs decimal ---")
for col in factor_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        continue
    vals = df[col].dropna()
    if len(vals) == 0:
        continue
    if vals.min() >= -1 and vals.max() <= 1:
        print(f"  {col:<35s} range [{vals.min():.6f}, {vals.max():.6f}] — decimal scale")
    elif vals.min() >= -100 and vals.max() <= 100 and vals.std() > 0.5:
        print(f"  {col:<35s} range [{vals.min():.4f}, {vals.max():.4f}] — possibly percentage")

# ── Constant or near-constant ────────────────────────────────────────────────
print(f"\n--- Constant or near-constant columns ---")
any_const = False
for col in factor_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        continue
    vals = df[col].dropna()
    if len(vals) == 0:
        continue
    if vals.nunique() <= 3:
        any_const = True
        print(f"  {col:<35s} only {vals.nunique()} unique values: {sorted(vals.unique()[:5])}")
    elif abs(vals.mean()) > 1e-10 and vals.std() / abs(vals.mean()) < 0.001:
        any_const = True
        print(f"  {col:<35s} near-constant (cv = {vals.std()/abs(vals.mean()):.6f})")

if not any_const:
    print(f"  ✓ No constant or near-constant columns")

# ── Check for overlap with macro_daily ───────────────────────────────────────
print(f"\n--- Potential overlap with macro_daily ---")
daily_factors = ['vix', 'mktrf', 'smb', 'hml', 'rmw', 'cma', 'rf', 'umd',
                 'fx_jpy', 'fx_eur', 'fx_gbp', 'widx_deu', 'widx_jpn']
for col in factor_cols:
    for dcol in daily_factors:
        if col.lower() == dcol.lower() or dcol.lower() in col.lower():
            print(f"  {col:<35s} may overlap with macro_daily.{dcol}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 5: SUMMARY — DECISIONS NEEDED
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 5: SUMMARY — DECISIONS NEEDED")
print("=" * 90)

print(f"""
Review the output above:

1. COLUMNS TO DROP:
   - Any column with ≥30% NaN
   - Constant or near-constant columns
   - Columns discontinued (long tail NaN to present)
   - Columns that duplicate macro_daily at lower frequency

2. PLACEHOLDER VALUES:
   - Replace identified placeholders with NaN

3. DATE RANGE:
   - Trim to 2004-01-01 if earlier data exists

4. NaN HANDLING:
   - Monthly macro data — forward-fill appropriate in merge pipeline
   - Leave as-is in cleaned file

Paste back the output and I will write the cleaning cell.
""")

STAGE 0: LOAD & INSPECT — Macro Monthly

  Shape: 252 rows × 23 columns
  Date range: 2004-01-30 → 2024-12-31
  Unique dates: 252
  Factor columns: 22

Columns and dtypes (22 factors):
    1. b30ret                              Float64        
    2. b30ind                              Float64        
    3. b20ret                              Float64        
    4. b20ind                              Float64        
    5. b10ret                              Float64        
    6. b10ind                              Float64        
    7. b7ret                               Float64        
    8. b7ind                               Float64        
    9. b5ret                               Float64        
   10. b5ind                               Float64        
   11. b2ret                               Float64        
   12. b2ind                               Float64        
   13. b1ret                               Float64        
   14. b1ind                               Float

In [2]:
# %% [markdown]
# ## Stage 6: Clean & Save
#
# **Data overview:**
# Monthly macro data from WRDS. 252 rows (21 years × 12 months), 22 factors.
# Contains Treasury bond returns and index levels across the maturity curve
# (1Y, 2Y, 5Y, 7Y, 10Y, 20Y, 30Y), T-bill returns (30-day, 90-day),
# CPI return and index, and Pastor-Stambaugh liquidity factor.
#
# **No columns dropped.** All 22 factors retained.
# **No rows dropped.** Perfect monthly coverage.
# **Zero NaN.** No missing data of any kind.
# **No placeholders detected.** No constant columns.
# **No winsorisation.** Applied in merge pipeline.
# **No overlap with macro_daily.** Entirely different variables.
#
# **Dates are last trading day of month** (70.2% are month-end, the rest are
# the last Friday of the month). Left as-is — the merge pipeline aligns
# via `merge_asof(direction='backward')`.
#
# **Factors retained: 22** (all kept)

# %%
print("=" * 90)
print("STAGE 6: CLEAN & SAVE")
print("=" * 90)

factor_cols_final = [c for c in df.columns if c != 'date']

print(f"\n  Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Factor columns: {len(factor_cols_final)}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  NaN: {df[factor_cols_final].isna().sum().sum()}")

print(f"\n  Factor list:")
for i, c in enumerate(factor_cols_final, 1):
    vals = df[c]
    print(f"    {i:>3d}. {c:<20s} range: [{vals.min():.4f}, {vals.max():.4f}]")

out_path = OUT_DIR / 'macro_monthly_clean.parquet'
df.to_parquet(out_path, index=False, engine='pyarrow')
print(f"\n  ✓ Saved: {out_path}")
print(f"    {df.shape[0]:,} rows × {df.shape[1]} columns ({len(factor_cols_final)} factors)")

print("\nCleaning complete.")

STAGE 6: CLEAN & SAVE

  Final shape: 252 rows × 23 columns
  Factor columns: 22
  Date range: 2004-01-30 → 2024-12-31
  NaN: 0

  Factor list:
      1. b30ret               range: [-0.1474, 0.1722]
      2. b30ind               range: [1149.6150, 4177.6270]
      3. b20ret               range: [-0.1059, 0.1445]
      4. b20ind               range: [1395.7090, 4583.5250]
      5. b10ret               range: [-0.0501, 0.0854]
      6. b10ind               range: [1281.8770, 2988.8110]
      7. b7ret                range: [-0.0405, 0.0819]
      8. b7ind                range: [1395.8400, 3090.1800]
      9. b5ret                range: [-0.0338, 0.0452]
     10. b5ind                range: [1261.4080, 2388.6270]
     11. b2ret                range: [-0.0151, 0.0207]
     12. b2ind                range: [1095.2830, 1652.6080]
     13. b1ret                range: [-0.0050, 0.0131]
     14. b1ind                range: [1025.5630, 1466.6800]
     15. t90ret               range: [-0.0001, 0.00